# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Access the metadata
metadata = dataset.metadata

# Print the dataset title and description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Each data element (record sets, fields, columns) is referenced by its `@id`.

In [ ]:
# List available record sets and their @id
record_sets = []
if hasattr(metadata, 'record_set'):
    record_sets = metadata.record_set

print('Record sets available:')
for rs in record_sets:
    print(f"- {rs['@id']}")

# Show fields for each record set
for rs in record_sets:
    print(f"\nFields in record set {rs['@id']}:")
    for field in rs.get('field', []):
        print(f"  - {field['@id']} ({field.get('name', 'Unnamed')})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set
dataframes = {}

# Collect all record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Print columns for each DataFrame
for record_set_id in record_set_ids:
    print(f"Columns for record set {record_set_id}: {dataframes[record_set_id].columns.tolist()}")

# Display the first rows for the primary record set (if present)
if record_set_ids:
    print(dataframes[record_set_ids[0]].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Choose record set to analyze
primary_record_set = record_set_ids[0] if record_set_ids else None
df = dataframes[primary_record_set] if primary_record_set else pd.DataFrame()

# List numeric columns (fields with type schema:Integer or schema:Float)
numeric_field_ids = []
for rs in record_sets:
    if rs['@id'] == primary_record_set:
        for field in rs.get('field', []):
            data_type = field.get('dataType', '')
            if data_type in ['schema:Integer', 'schema:Float']:
                numeric_field_ids.append(field['@id'])

print(f"Numeric fields (@id): {numeric_field_ids}")

# If available, select the first numeric field for EDA
if numeric_field_ids:
    numeric_field = numeric_field_ids[0]
    threshold = 10
    # Filter records with values greater than threshold
    if numeric_field in df.columns:
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"].head()])

        # Attempt to group by a demographic field (e.g., level_of_education or gender)
        # Find a categorical/grouping field
        group_field_candidates = []
        for field in rs.get('field', []):
            data_type = field.get('dataType', '')
            if data_type == 'schema:Text':
                group_field_candidates.append(field['@id'])
        print(f"Potential group fields: {group_field_candidates}")

        # If available, use the first group field
        if group_field_candidates:
            group_field = group_field_candidates[0]
            if group_field in filtered_df.columns:
                grouped_df = (
                    filtered_df.groupby(group_field)[numeric_field]
                    .mean()
                    .reset_index()
                )
                print(f"Grouped data by {group_field} (mean {numeric_field}):")
                print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt

# Plot histogram for the numeric field (if available)
if numeric_field_ids and primary_record_set:
    numeric_field = numeric_field_ids[0]
    if numeric_field in df.columns:
        plt.figure(figsize=(8, 6))
        df[numeric_field].dropna().hist(bins=20)
        plt.xlabel(numeric_field)
        plt.ylabel('Frequency')
        plt.title(f'Distribution of {numeric_field} in {primary_record_set}')
        plt.show()

        # If grouping field available, plot group means
        if 'grouped_df' in locals():
            plt.figure(figsize=(8, 6))
            plt.bar(grouped_df[group_field], grouped_df[numeric_field])
            plt.xlabel(group_field)
            plt.ylabel(f'Mean {numeric_field}')
            plt.title(f'Mean {numeric_field} by {group_field} in {primary_record_set}')
            plt.xticks(rotation=45)
            plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset contains rich survey records for demographics and mental health indicators.
- Numeric fields (e.g., GAD-7, PHQ-9 scores) facilitate quantitative analysis, including normalization and grouping.
- Demographic fields (e.g., level_of_education, gender) allow stratification and visualization.
- Using `mlcroissant`, we can efficiently reference entities using their `@id` for robust, reproducible workflows.